# 07 · Glides: descending spectral ridges

After the acid, a band in the **excess spectrogram** (each frame divided by a quiet baseline spectrum) may
descend smoothly in frequency (Sep 2026: 6.5 → 0.9 kHz in 328 s on one hydrophone, never on the air mic). This
notebook tracks it per channel, tests whether it is real (contrast), checks harmonics, and follows it across takes.

Read `docs/STUDENT_GUIDE.md` §13. The mechanism is not established, and a Minnaert radius read off the ridge
is shown only with its Bond number, as a feasibility check.

In [ ]:
# ==== YOUR SETTINGS ====
CONFIG = "../config/demo.yaml"
TAKE = "DEMO_002"
BASE_TAKE = "DEMO_002"        # take that holds the quiet baseline (same sensors, same geometry)
BASELINE = (5.0, 34.0)        # [s] baseline window inside BASE_TAKE
T0 = 46.0                     # [s] start tracking here (after the onset)
SEED = (4000.0, 12000.0)      # [Hz] where to pick up the ridge at T0
FMAX = 20000
LATER_TAKES = ["DEMO_003", "DEMO_004"]   # follow the ridge into later takes (baseline from BASE_TAKE)

In [ ]:
# ---- standard setup (same in every notebook) ----
%matplotlib inline
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import zoom_acoustics as za
from zoom_acoustics import plots as zp, dsp

cfg = za.load_config(CONFIG)
print("config   :", cfg.config_file)
print("data_dir :", cfg.data_dir)
print("cache_dir:", cfg.cache_dir)
print("figures  :", cfg.figures_dir)

In [ ]:
from zoom_acoustics import glide as G
f = za.load_features(cfg, TAKE)
fb = za.load_features(cfg, BASE_TAKE)
tracks, rows = {}, []
for c in f.channels:
    tr = G.track_ridge(f, c, BASELINE, T0, seed=SEED, base_feat=fb)
    s = G.summarize(tr)
    if s:
        tracks[c] = tr
        rows.append(dict(channel=c, label=f.label(c), **s))
summary = pd.DataFrame(rows)
summary[["channel", "label", "f_start_hz", "f_end_hz", "octaves", "octaves_per_min",
         "median_contrast_db", "significant", "t_lock_s", "frac_locked",
         "minnaert_R_end_mm", "bond_end", "free_bubble_feasible_at_end"]].round(3)

**Read `significant` first.** A tracker always returns a path, even through pure noise, and a noise path drifts.
Only tracks with median contrast well above ~3 dB are ridges. The air mic is the control: if it "glides" too with
high contrast, the effect is not liquid-borne. `t_lock_s` later than `T0` means the seed started on something else.

In [ ]:
show = int(summary.sort_values("median_contrast_db").channel.iloc[-1]) if len(summary) else f.channels[0]
fig = zp.plot_glide(f, tracks, BASELINE, show_ch=show, fmax=FMAX, base_feat=fb)
zp.save(fig, cfg, f"{TAKE}_glide")

## Harmonics along the ridge

Is there a line at m × f(t)? Non-integer multiples (1.5×, 2.5×) are the noise controls. A strong 0.5× means the
tracked ridge is itself a **harmonic**: re-seed lower.

In [ ]:
G.harmonic_ladder(f, show, BASELINE, tracks[show], base_feat=fb).round(2)

## Following the ridge into later takes

Each later take is tracked with the baseline of `BASE_TAKE`, seeded near where the ridge was last seen. Only
valid if nothing was moved or re-gained between takes.

In [ ]:
all_rows = [dict(take=TAKE, **r) for r in rows]
last = {r["channel"]: r["f_end_hz"] for r in rows}
for name in LATER_TAKES:
    fl = za.load_features(cfg, name)
    for c in fl.channels:
        if c not in last:
            continue
        tr = G.track_ridge(fl, c, BASELINE, 1.0, seed=(0.7 * last[c], 1.3 * last[c]), base_feat=fb)
        s = G.summarize(tr)
        if s:
            all_rows.append(dict(take=name, channel=c, label=fl.label(c), **s))
            if s["significant"]:
                last[c] = s["f_end_hz"]
tab = pd.DataFrame(all_rows)
tab[["take", "channel", "f_start_hz", "f_end_hz", "octaves", "median_contrast_db", "significant"]].round(2)

In [ ]:
tab.to_csv(cfg.figures_path / "glide_summary.csv", index=False)
print("wrote", cfg.figures_path / "glide_summary.csv")